#### Análise Exploratória de Dados: E-commerce Brasileiro (Olist)

**Objetivo:** Este notebook apresenta uma análise exploratória do conjunto de dados público da Olist, a maior loja de departamentos em marketplaces brasileiros. O foco é extrair insights acionáveis sobre logística, comportamento de pagamento, desempenho de produtos e satisfação do cliente para responder a perguntas estratégicas de negócio.

**Autores:** Kaike Brito Leitão, Enrico Santos Navajas e Mario

```mermaid
erDiagram
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        string customer_zip_code_prefix FK
        string customer_city
        string customer_state
    }
    
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        datetime order_purchase_timestamp
        datetime order_approved_at
        datetime order_delivered_carrier_date
        datetime order_delivered_customer_date
        datetime order_estimated_delivery_date
    }
    
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        datetime shipping_limit_date
        float price
        float freight_value
    }
    
    PRODUCTS {
        string product_id PK
        string product_category_name FK
        int product_name_lenght
        int product_description_lenght
        int product_photos_qty
        float product_weight_g
        float product_length_cm
        float product_height_cm
        float product_width_cm
    }
    
    SELLERS {
        string seller_id PK
        string seller_zip_code_prefix FK
        string seller_city
        string seller_state
    }
    
    ORDER_PAYMENTS {
        string order_id PK, FK
        int payment_sequential PK
        string payment_type
        int payment_installments
        float payment_value
    }
    
    ORDER_REVIEWS {
        string review_id PK
        string order_id FK
        int review_score
        string review_comment_title
        string review_comment_message
        datetime review_creation_date
        datetime review_answer_timestamp
    }
    
    GEOLOCATION {
        string geolocation_zip_code_prefix PK
        float geolocation_lat
        float geolocation_lng
        string geolocation_city
        string geolocation_state
    }
    
    TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }

    %% Relacionamentos mapeados para a regra de negócio %%
    CUSTOMERS ||--o{ ORDERS : "realiza_pedido"
    ORDERS ||--|{ ORDER_ITEMS : "contem_itens"
    ORDERS ||--|{ ORDER_PAYMENTS : "possui_pagamentos"
    ORDERS ||--o{ ORDER_REVIEWS : "recebe_avaliacao"
    PRODUCTS ||--o{ ORDER_ITEMS : "compoe"
    SELLERS ||--o{ ORDER_ITEMS : "fornece"
    GEOLOCATION ||--o{ CUSTOMERS : "localiza_cliente"
    GEOLOCATION ||--o{ SELLERS : "localiza_vendedor"
    TRANSLATION ||--o{ PRODUCTS : "traduz_categoria"


In [2]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import logging
from pathlib import Path
from typing import Dict, Tuple, List
from IPython.display import display
from IPython.display import display
 
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [3]:
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [4]:
# ── Constantes ────────────────────────────────────────────────────────────────
RAW          = Path("../dataframes/raw/")           # ajustar para o seu path local
OUT          = Path("../dataframes/processed/")
OUT.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE    = 0.20
OUTLIER_P99  = 46    # dias — P99 calculado nos dados reais
PALETA       = "#2563EB"
 
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

In [5]:
# =============================================================================
# SEÇÃO 1 — CARREGAMENTO DAS TABELAS
# =============================================================================
 
def carregar_tabelas(path: Path) -> Dict[str, pd.DataFrame]:
    """
    Carrega os 9 CSVs do Olist em um dicionário tipado.
    
    Args:
        path: Caminho da pasta contendo os CSVs.
    Returns:
        Dicionário {nome_tabela: DataFrame}
    """
    arquivos = {
        "orders":      "olist_orders_dataset.csv",
        "order_items": "olist_order_items_dataset.csv",
        "payments":    "olist_order_payments_dataset.csv",
        "products":    "olist_products_dataset.csv",
        "sellers":     "olist_sellers_dataset.csv",
        "customers":   "olist_customers_dataset.csv",
        "geolocation": "olist_geolocation_dataset.csv",
        "translation": "product_category_name_translation.csv",
    }
    dfs = {}
    for nome, arq in arquivos.items():
        dfs[nome] = pd.read_csv(path / arq)
        logger.info(f"✅ {nome}: {dfs[nome].shape}")
    return dfs
 
dfs = carregar_tabelas(RAW)

INFO | ✅ orders: (99441, 8)
INFO | ✅ order_items: (112650, 7)
INFO | ✅ payments: (103886, 5)
INFO | ✅ products: (32951, 9)
INFO | ✅ sellers: (3095, 4)
INFO | ✅ customers: (99441, 5)
INFO | ✅ geolocation: (1000163, 5)
INFO | ✅ translation: (71, 2)


In [6]:
# =============================================================================
# SEÇÃO 2 — ENGENHARIA DE FEATURES ENRIQUECIDA
# =============================================================================
# Cada linha do dataset final = 1 pedido entregue
# Tabelas utilizadas: orders (hub) + order_items + payments + products +
#                     sellers + customers + geolocation + translation
#
# NOVAS FEATURES vs. versão anterior:
#   Geográficas  → dist_km (Haversine), delta_lat, delta_lng, mesma_uf, mesma_regiao
#                  regiao_cliente, regiao_vendedor, media_dias_uf_cliente
#   Logísticas   → seller_avg_delivery, seller_std_delivery, seller_n_orders
#                  fim_de_semana, periodo_dia, faixa_dist_km
#   Produto      → densidade_g_cm3, freight_ratio, avg_price_per_item, price_range
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict

# ── Mapeamento de UF → Macrorregião ──────────────────────────────────────────
REGIAO_MAP: Dict[str, str] = {
    "AC":"Norte","AM":"Norte","AP":"Norte","PA":"Norte",
    "RO":"Norte","RR":"Norte","TO":"Norte",
    "AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste",
    "PB":"Nordeste","PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MS":"Centro-Oeste","MT":"Centro-Oeste",
    "ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste",
    "PR":"Sul","RS":"Sul","SC":"Sul",
}

def haversine_vec(lat1: np.ndarray, lon1: np.ndarray,
                   lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    """
    Distância geodésica em km entre dois pontos geográficos (vetorizada).

    Fórmula de Haversine:
        a = sin²(Δlat/2) + cos(lat1)·cos(lat2)·sin²(Δlon/2)
        d = 2R · arcsin(√a)   onde R = 6371 km (raio médio da Terra)

    Por que Haversine e não distância euclidiana?
    A distância euclidiana em graus (√(Δlat²+Δlon²)) ignora que a Terra é
    esférica e que 1° de longitude vale distâncias diferentes dependendo da
    latitude. No Brasil, onde latitudes vão de -33° a +5°, o erro euclidiano
    pode chegar a 15% em rotas longas (ex: SP→AM). A Haversine resolve isso
    com custo computacional praticamente igual.

    Args:
        lat1, lon1: Arrays de coordenadas do ponto de origem (vendedor).
        lat2, lon2: Arrays de coordenadas do ponto de destino (cliente).

    Returns:
        Array de distâncias em km.
    """
    R = 6371.0
    lat1, lon1, lat2, lon2 = (
        np.radians(lat1), np.radians(lon1),
        np.radians(lat2), np.radians(lon2),
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def construir_dataset(dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Integra as 8 tabelas em um único DataFrame de modelagem, com feature
    engineering completo e enriquecido.

    Sumário de features geradas:
    ┌─ Temporais      ─┐  estimativa_prazo, dia_semana_compra, hora_compra,
    │                   │  mes_compra, fim_de_semana, periodo_dia
    ├─ Geográficas    ─┤  dist_km*, delta_lat*, delta_lng*, mesma_uf*,
    │                   │  mesma_regiao*, regiao_cliente, regiao_vendedor,
    │                   │  media_dias_uf_cliente*, geolocation_lat/lng
    ├─ Logísticas     ─┤  dias_ate_aprova_h, n_items, n_sellers,
    │                   │  faixa_dist_km*, freight_ratio*, avg_price_per_item*
    ├─ Seller         ─┤  seller_avg_delivery*, seller_std_delivery*,
    │                   │  seller_n_orders*
    ├─ Produto        ─┤  product_weight_g, volume_cm3, densidade_g_cm3*,
    │                   │  product_photos_qty
    └─ Pagamento      ─┘  payment_type, payment_value_total,
                           payment_installments_max, price_range*, freight_total

    * = feature nova em relação à versão anterior

    Correlações validadas com target (dias_entrega):
        dist_km            r = 0.39  ← mais forte das geográficas
        estimativa_prazo   r = 0.38
        mesma_uf           r = -0.36 (negativa: mesma UF → entrega mais rápida)
        seller_avg_delivery r = 0.35
        delta_lat          r = 0.22
        freight_total      r = 0.17
        seller_std_delivery r = 0.22

    Returns:
        DataFrame bruto (antes da limpeza de outliers). Shape esperado: ~96k linhas.
    """

    # ── [1] ORDERS: filtrar entregues + datas ─────────────────────────────────
    orders = dfs["orders"].copy()
    for col in ["order_purchase_timestamp", "order_delivered_customer_date",
                "order_estimated_delivery_date", "order_approved_at",
                "order_delivered_carrier_date"]:
        orders[col] = pd.to_datetime(orders[col])

    df = orders[orders["order_status"] == "delivered"].copy()
    df = df.dropna(subset=["order_delivered_customer_date"])

    # ── [2] TARGET ────────────────────────────────────────────────────────────
    df["dias_entrega"] = (
        df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
    ).dt.days

    # ── [3] FEATURES TEMPORAIS ────────────────────────────────────────────────
    df["estimativa_prazo"]  = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days
    df["dia_semana_compra"] = df["order_purchase_timestamp"].dt.dayofweek   # 0=Seg, 6=Dom
    df["hora_compra"]       = df["order_purchase_timestamp"].dt.hour
    df["mes_compra"]        = df["order_purchase_timestamp"].dt.month
    df["dias_ate_aprova_h"] = (
        df["order_approved_at"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

    # NOVA: fim_de_semana — pedidos no fim de semana tendem a demorar mais para
    # aprovação pois os processos bancários ficam suspensos (sábado=5, domingo=6)
    df["fim_de_semana"] = df["dia_semana_compra"].isin([5, 6]).astype(int)

    # NOVA: periodo_dia — compras à noite/madrugada são processadas no próximo
    # dia útil, podendo atrasar a aprovação e o início da preparação do pedido
    df["periodo_dia"] = pd.cut(
        df["hora_compra"],
        bins=[0, 6, 12, 18, 24],
        labels=[0, 1, 2, 3],    # 0=madrugada, 1=manhã, 2=tarde, 3=noite
        right=False
    ).astype(float)

    df = df[["order_id", "customer_id", "dias_entrega", "estimativa_prazo",
             "dia_semana_compra", "hora_compra", "mes_compra", "dias_ate_aprova_h",
             "fim_de_semana", "periodo_dia"]]

    # ── [4] ORDER ITEMS → 1 linha por pedido ─────────────────────────────────
    items = dfs["order_items"]
    items_agg = items.groupby("order_id").agg(
        n_items         = ("order_item_id", "count"),
        price_total     = ("price",         "sum"),
        freight_total   = ("freight_value", "sum"),
        n_sellers       = ("seller_id",     "nunique"),
        price_max       = ("price",         "max"),
        price_min       = ("price",         "min"),
        product_id_1st  = ("product_id",    "first"),
        seller_id_1st   = ("seller_id",     "first"),
    ).reset_index()

    # NOVA: price_range — amplitude de preços no pedido (pedidos com itens
    # de preços muito diferentes tendem a ter logística mais complexa)
    items_agg["price_range"] = items_agg["price_max"] - items_agg["price_min"]

    # NOVA: freight_ratio — proporção do frete em relação ao valor do pedido.
    # Fretes altos relativos ao produto indicam distância ou peso elevado.
    items_agg["freight_ratio"] = (
        items_agg["freight_total"] / (items_agg["price_total"] + 0.01)
    ).clip(upper=5.0)

    # NOVA: avg_price_per_item — ticket médio por item do pedido
    items_agg["avg_price_per_item"] = items_agg["price_total"] / items_agg["n_items"]

    df = df.merge(items_agg.drop(columns=["price_max", "price_min"]),
                  on="order_id", how="left")

    # ── [5] PAGAMENTOS ────────────────────────────────────────────────────────
    pay = dfs["payments"]
    pay_agg = pay.groupby("order_id").agg(
        payment_value_total      = ("payment_value",       "sum"),
        payment_installments_max = ("payment_installments", "max"),
    ).reset_index()
    pay_type = (pay.sort_values("payment_sequential")
                   .groupby("order_id")["payment_type"].first().reset_index())
    pay_agg = pay_agg.merge(pay_type, on="order_id")
    df = df.merge(pay_agg, on="order_id", how="left")

    # ── [6] PRODUTOS + TRADUÇÃO ───────────────────────────────────────────────
    prod = dfs["products"].merge(dfs["translation"], on="product_category_name", how="left")
    prod["volume_cm3"] = (
        prod["product_length_cm"] * prod["product_height_cm"] * prod["product_width_cm"]
    )
    # NOVA: densidade_g_cm3 — relação peso/volume. Produtos densos (metais,
    # ferramentas) têm custo de frete diferente de produtos volumosos mas leves
    # (almofadas, brinquedos). Densidade capta o que nem peso nem volume captam sozinhos.
    prod["densidade_g_cm3"] = (
        prod["product_weight_g"] / prod["volume_cm3"].replace(0, np.nan)
    ).clip(upper=10.0)

    df = df.merge(
        prod[["product_id", "product_category_name_english", "product_weight_g",
              "volume_cm3", "densidade_g_cm3", "product_photos_qty"]],
        left_on="product_id_1st", right_on="product_id", how="left"
    )

    # ── [7] GEOLOCALIZAÇÃO — mediana por ZIP (reduz ruído GPS) ───────────────
    geo = dfs["geolocation"]
    geo_med = (
        geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
           .median().reset_index()
    )

    # Coordenadas do CLIENTE
    cust_geo = (dfs["customers"][["customer_id", "customer_zip_code_prefix"]]
                .merge(geo_med, left_on="customer_zip_code_prefix",
                       right_on="geolocation_zip_code_prefix", how="left")
                .rename(columns={"geolocation_lat": "clat", "geolocation_lng": "clng"}))

    # Coordenadas do VENDEDOR PRINCIPAL do pedido
    sell_geo = (dfs["sellers"][["seller_id", "seller_zip_code_prefix"]]
                .merge(geo_med, left_on="seller_zip_code_prefix",
                       right_on="geolocation_zip_code_prefix", how="left")
                .rename(columns={"geolocation_lat": "slat", "geolocation_lng": "slng"}))

    df = df.merge(cust_geo[["customer_id", "clat", "clng"]], on="customer_id", how="left")
    df = df.merge(sell_geo[["seller_id", "slat", "slng"]],
                  left_on="seller_id_1st", right_on="seller_id", how="left")

    # NOVA: dist_km — distância geodésica (Haversine) entre cliente e vendedor.
    # É a feature geográfica mais informativa: r = 0.39 com o target.
    # Substitui e supera o uso isolado de lat/lng do cliente.
    valid_geo = (
        df["clat"].notna() & df["slat"].notna() &
        df["clat"].between(-35, 6) & df["slat"].between(-35, 6)
    )
    df["dist_km"] = np.nan
    df.loc[valid_geo, "dist_km"] = haversine_vec(
        df.loc[valid_geo, "clat"].values, df.loc[valid_geo, "clng"].values,
        df.loc[valid_geo, "slat"].values, df.loc[valid_geo, "slng"].values,
    )

    # NOVA: delta_lat / delta_lng — diferença de coordenadas entre cliente e
    # vendedor. Capturam a direção da rota (Norte-Sul vs. Leste-Oeste), que
    # a distância escalar não captura. Ex.: AM→SP (delta_lat grande) tem
    # prazo diferente de PE→BA (delta_lat similar mas rota mais curta).
    df["delta_lat"] = df["clat"] - df["slat"]
    df["delta_lng"] = df["clng"] - df["slng"]

    # Expor lat/lng do cliente para o modelo (feature original, mantida)
    df = df.rename(columns={"clat": "geolocation_lat", "clng": "geolocation_lng"})

    # NOVA: faixa_dist_km — categorização da distância em faixas operacionais.
    # Operadoras logísticas têm contratos e SLAs diferentes por faixa de distância.
    df["faixa_dist_km"] = pd.cut(
        df["dist_km"],
        bins=[0, 100, 300, 600, 1000, 2000, 10000],
        labels=[0, 1, 2, 3, 4, 5],   # 0=local, 1=regional, 2=inter-regional,
        right=False                    # 3=longa, 4=muito longa, 5=extrema
    ).astype(float)

    # ── [8] VENDEDORES + FEATURES GEOGRÁFICAS UF ─────────────────────────────
    df = df.merge(
        dfs["sellers"][["seller_id", "seller_state"]],
        left_on="seller_id_1st", right_on="seller_id", how="left",
        suffixes=("", "_dup")
    )
    df = df.drop(columns=[c for c in df.columns if c.endswith("_dup")], errors="ignore")

    # ── [9] CLIENTES ──────────────────────────────────────────────────────────
    df = df.merge(
        dfs["customers"][["customer_id", "customer_state", "customer_zip_code_prefix"]],
        on="customer_id", how="left"
    )

    # NOVA: mesma_uf — flag binária: cliente e vendedor principal na mesma UF?
    # Pedidos intraestaduais chegam em média 7,5 dias vs. 14,7 dias (interestaduais).
    # Correlação com target: r = -0.36 (a mais forte feature binária do dataset).
    df["mesma_uf"] = (df["customer_state"] == df["seller_state"]).astype(int)

    # NOVA: mesma_regiao — cliente e vendedor na mesma macrorregião do Brasil?
    # (Norte / Nordeste / Centro-Oeste / Sudeste / Sul)
    # Captura padrão intermediário entre mesma_uf e dist_km.
    df["regiao_cliente"]  = df["customer_state"].map(REGIAO_MAP)
    df["regiao_vendedor"] = df["seller_state"].map(REGIAO_MAP)
    df["mesma_regiao"]    = (df["regiao_cliente"] == df["regiao_vendedor"]).astype(int)

    # ── [10] HISTÓRICO DO VENDEDOR (Leave-One-Out safe para produção) ─────────
    # Intuição: vendedores com histórico de entregas mais rápidas tendem a
    # continuar entregando rápido — captura qualidade operacional do seller.
    # ATENÇÃO: em produção, calcular seller_avg_delivery APENAS com pedidos
    # ANTERIORES à data do pedido atual (para evitar leakage temporal).
    seller_stats = (
        df[["seller_id_1st", "dias_entrega"]]
        .groupby("seller_id_1st")
        .agg(
            seller_avg_delivery = ("dias_entrega", "mean"),
            seller_std_delivery = ("dias_entrega", "std"),
            seller_n_orders     = ("dias_entrega", "count"),
        )
        .reset_index()
        .rename(columns={"seller_id_1st": "seller_id_stat"})
    )
    df = df.merge(
        seller_stats,
        left_on="seller_id_1st", right_on="seller_id_stat", how="left"
    )
    # Preencher sellers com apenas 1 pedido (std = NaN → mediana global)
    df["seller_std_delivery"] = df["seller_std_delivery"].fillna(
        df["seller_std_delivery"].median()
    )

    # NOVA: media_dias_uf_cliente — tempo médio histórico de entrega para a UF
    # do cliente. Captura a infraestrutura logística da região de destino.
    uf_stats = (
        df.groupby("customer_state")["dias_entrega"]
        .mean()
        .rename("media_dias_uf_cliente")
    )
    df = df.merge(uf_stats, on="customer_state", how="left")

    # ── [11] REMOVER COLUNAS AUXILIARES ───────────────────────────────────────
    df = df.drop(columns=[
        "product_id", "seller_id", "seller_id_dup",
        "product_id_1st", "seller_id_1st", "seller_id_stat",
        "customer_zip_code_prefix", "customer_id", "order_id",
        "slat", "slng",
    ], errors="ignore")

    return df.reset_index(drop=True)


def exibir_tabela(df: pd.DataFrame, titulo: str = None) -> None:
    if titulo:
        print(f"\n📋 {titulo}")
    display(df)


df_raw = construir_dataset(dfs)

feature_summary = pd.DataFrame([
    {"Domínio": "Temporais",   "Features": "estimativa_prazo, dia_semana_compra, hora_compra, mes_compra, fim_de_semana, periodo_dia"},
    {"Domínio": "Geográficas", "Features": "dist_km, delta_lat, delta_lng, mesma_uf, mesma_regiao, regiao_cliente, regiao_vendedor, faixa_dist_km, geolocation_lat, geolocation_lng, media_dias_uf_cliente, customer_state, seller_state"},
    {"Domínio": "Logísticas",  "Features": "dias_ate_aprova_h, n_items, n_sellers, freight_total, freight_ratio, avg_price_per_item, price_range, price_total"},
    {"Domínio": "Seller",      "Features": "seller_avg_delivery, seller_std_delivery, seller_n_orders"},
    {"Domínio": "Produto",     "Features": "product_weight_g, volume_cm3, densidade_g_cm3, product_photos_qty, product_category_name_english"},
    {"Domínio": "Pagamento",   "Features": "payment_type, payment_value_total, payment_installments_max"},
])
print("\n📌 Sumário de features geradas")
exibir_tabela(feature_summary)


📌 Sumário de features geradas


,Domínio,Features
0,Temporais,"estimativa_prazo, dia_semana_compra, hora_comp..."
1,Geográficas,"dist_km, delta_lat, delta_lng, mesma_uf, mesma..."
2,Logísticas,"dias_ate_aprova_h, n_items, n_sellers, freight..."
3,Seller,"seller_avg_delivery, seller_std_delivery, sell..."
4,Produto,"product_weight_g, volume_cm3, densidade_g_cm3,..."
5,Pagamento,"payment_type, payment_value_total, payment_ins..."


In [7]:
# =============================================================================
# SEÇÃO 3 — FILTRAGEM E LIMPEZA DOS DADOS
# =============================================================================
 
def limpar_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove ruídos, inconsistências e outliers extremos.
 
    Regras aplicadas:
    1. Remove linhas com target nulo (8 linhas — dados incompletos de sistema)
    2. Remove entregas de 0 dias (impossível operacionalmente — erro de timestamp)
    3. Remove outliers acima do P99 (46 dias): eventos excepcionais que distorceriam
       os modelos. P99 calculado empiricamente nos dados reais.
    4. Remove estimativa_prazo nula ou não positiva (dados corrompidos)
 
    Returns:
        DataFrame limpo, reindexado.
    """
    n_inicial = len(df)
 
    df = df.dropna(subset=["dias_entrega"])           # 8 linhas com target nulo
    df = df[df["dias_entrega"] > 0]                   # remove 0 dias (erro de data)
    df = df[df["dias_entrega"] <= OUTLIER_P99]        # remove outliers acima do P99
    df = df.dropna(subset=["estimativa_prazo"])
    df = df[df["estimativa_prazo"] > 0]
 
    n_final = len(df)
    logger.info(f"Limpeza: {n_inicial:,} → {n_final:,} linhas ({n_inicial-n_final:,} removidas)")
    return df.reset_index(drop=True)
 
df_clean = limpar_dataset(df_raw)
 
print(f"✅ Dataset limpo: {df_clean.shape}")
target_stats = df_clean["dias_entrega"].describe().round(2).to_frame().T
target_stats["skewness"] = df_clean["dias_entrega"].skew().round(3)
target_stats.index = ["dias_entrega"]
print("\n📊 Estatísticas do target:")
exibir_tabela(target_stats)
 

INFO | Limpeza: 96,470 → 95,577 linhas (893 removidas)


✅ Dataset limpo: (95577, 39)

📊 Estatísticas do target:


,count,mean,std,min,25%,50%,75%,max,skewness
dias_entrega,95577.0,11.62,7.81,1.0,6.0,10.0,15.0,46.0,1.429


In [ ]:
# =============================================================================
# DOWNLOAD DO DATASET LIMPO
# =============================================================================
# Exporta o dataset final (após feature engineering + limpeza) para CSV,
# permitindo reuso em outros notebooks sem re-executar todo o pipeline.
# O arquivo inclui target + todas as 38 features enriquecidas.
 
_EXPORT_PATH = OUT / "olist_dataset_enriquecido.csv"
df_clean.to_csv(_EXPORT_PATH, index=False)
print(f"\n💾 Dataset exportado → {_EXPORT_PATH}")
print(f"   Shape: {df_clean.shape[0]:,} linhas × {df_clean.shape[1]} colunas")
print(f"   Tamanho: {_EXPORT_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"\n   Para carregar em outro notebook:")
print(f"   df_clean = pd.read_csv('{_EXPORT_PATH}')")

AttributeError: 'DataFrame' object has no attribute 'display'

In [ ]:
# =============================================================================
# SEÇÃO 5 — PRÉ-PROCESSAMENTO  (substituir a seção 5 anterior por este bloco)
# =============================================================================
#
# CAUSA DO KeyError ANTERIOR:
# A lista NUM_FEATURES usava as 16 features da versão antiga do dataset.
# O dataset enriquecido tem 38 colunas novas que não estavam listadas.
# Este bloco está sincronizado exatamente com as colunas de df_clean.
#
# Colunas disponíveis em df_clean (39 total = 1 target + 38 features):
# dias_entrega | estimativa_prazo | dia_semana_compra | hora_compra |
# mes_compra | dias_ate_aprova_h | fim_de_semana | periodo_dia |
# n_items | price_total | freight_total | n_sellers | price_range |
# freight_ratio | avg_price_per_item | payment_value_total |
# payment_installments_max | payment_type | product_category_name_english |
# product_weight_g | volume_cm3 | densidade_g_cm3 | product_photos_qty |
# geolocation_lat | geolocation_lng | dist_km | delta_lat | delta_lng |
# faixa_dist_km | seller_state | customer_state | mesma_uf |
# regiao_cliente | regiao_vendedor | mesma_regiao |
# seller_avg_delivery | seller_std_delivery | seller_n_orders |
# media_dias_uf_cliente

# ── 5.1 Definição das features por tipo ──────────────────────────────────────
TARGET = "dias_entrega"

NUM_FEATURES: List[str] = [
    # ── Temporais ──────────────────────────────────────────────────────────────
    "estimativa_prazo",           # r=+0.43 — feature mais correlacionada com target
    "dias_ate_aprova_h",          # r=+0.10 — demora na aprovação indica problema
    "dia_semana_compra",          # padrão de dia da semana (0=Seg, 6=Dom)
    "hora_compra",                # padrão de hora do dia (0–23)
    "mes_compra",                 # sazonalidade mensal (1–12)
    "fim_de_semana",              # 1 se sáb/dom — atraso no processamento bancário
    "periodo_dia",                # 0=madrugada, 1=manhã, 2=tarde, 3=noite
    # ── Financeiras ────────────────────────────────────────────────────────────
    "price_total",                # valor total dos produtos (R$)
    "freight_total",              # r=+0.18 — frete alto = produto pesado/distante
    "payment_value_total",        # valor total pago (inclui juros se parcelado)
    "payment_installments_max",   # parcelamento máximo (proxy de valor do pedido)
    "freight_ratio",              # r=+0.10 — frete ÷ preço (proxy de distância)
    "avg_price_per_item",         # ticket médio por item do pedido
    "price_range",                # amplitude de preços no pedido (max − min)
    # ── Itens ──────────────────────────────────────────────────────────────────
    "n_items",                    # quantidade de itens no pedido
    "n_sellers",                  # número de vendedores distintos no pedido
    # ── Produto ────────────────────────────────────────────────────────────────
    "product_weight_g",           # r=+0.08 — peso do produto principal (gramas)
    "volume_cm3",                 # volume do produto (L × A × C em cm³)
    "densidade_g_cm3",            # densidade g/cm³ — tipologia do produto
    "product_photos_qty",         # quantidade de fotos (proxy de qualidade)
    # ── Geográficas numéricas ──────────────────────────────────────────────────
    "geolocation_lat",            # r=+0.28 — latitude do cliente (mediana por CEP)
    "geolocation_lng",            # longitude do cliente (mediana por CEP)
    "dist_km",                    # r=+0.44 — distância Haversine cliente↔vendedor
    "delta_lat",                  # r=+0.23 — componente Norte-Sul da rota
    "delta_lng",                  # r=+0.12 — componente Leste-Oeste da rota
    "faixa_dist_km",              # r=+0.47 — faixa operacional (0=local … 5=extrema)
    "mesma_uf",                   # r=−0.41 — 1 se cliente e vendedor na mesma UF
    "mesma_regiao",               # r=−0.33 — 1 se mesma macrorregião do Brasil
    "media_dias_uf_cliente",      # r=+0.46 — média histórica de entrega para a UF
    # ── Seller ─────────────────────────────────────────────────────────────────
    "seller_avg_delivery",        # r=+0.35 — média histórica de entrega do vendedor
    "seller_std_delivery",        # r=+0.15 — variabilidade histórica do vendedor
    "seller_n_orders",            # volume histórico de pedidos do vendedor
]

CAT_FEATURES: List[str] = [
    "payment_type",                    # credit_card / boleto / voucher / debit_card
    "customer_state",                  # 27 UFs — infraestrutura de destino
    "seller_state",                    # UF do vendedor — infraestrutura de origem
    "regiao_cliente",                  # Norte / Nordeste / Centro-Oeste / Sudeste / Sul
    "regiao_vendedor",                 # macrorregião do vendedor principal
    "product_category_name_english",   # 71 categorias de produto
]

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES

# ── Recarga automática do dataset enriquecido ─────────────────────────────────
# Se df_clean ainda tem as 21 colunas da versão antiga da construir_dataset,
# recarrega automaticamente do CSV exportado na Seção 3 (39 colunas).
# Isso acontece quando a Seção 2 do notebook ainda usa a função antiga.
_colunas_esperadas = set(ALL_FEATURES + [TARGET])
if not _colunas_esperadas.issubset(set(df_clean.columns)):
    _csv_enriquecido = OUT / "olist_dataset_enriquecido.csv"
    print(f"⚠️  df_clean incompleto ({df_clean.shape[1]} colunas detectadas).")
    print(f"   Recarregando dataset enriquecido de:\n   {_csv_enriquecido}")
    df_clean = pd.read_csv(_csv_enriquecido)
    print(f"✅ df_clean recarregado: {df_clean.shape[0]:,} linhas × {df_clean.shape[1]} colunas")

# Verificação final
_faltando = [f for f in ALL_FEATURES if f not in df_clean.columns]
if _faltando:
    raise KeyError(
        f"Features ainda ausentes após recarga — verifique o CSV:\n{_faltando}"
    )

X = df_clean[ALL_FEATURES].copy()
y = df_clean[TARGET].copy()

print(f"\n📐 Features: {len(ALL_FEATURES)} total ({len(NUM_FEATURES)} numéricas + {len(CAT_FEATURES)} categóricas)")
print(f"📐 X shape: {X.shape} | y shape: {y.shape}")

# ── 5.2 Split treino/teste ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
print(f"\n✂️  Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")

# ── 5.3 Análise de nulos ──────────────────────────────────────────────────────
nulos = X_train.isnull().sum()
nulos_pct = (nulos[nulos > 0] / len(X_train) * 100).round(2)

nulos_df = (nulos_pct
            .reset_index()
            .rename(columns={"index": "feature", 0: "pct_nulos_%"}))

_estrategia_cat = {"product_category_name_english", "regiao_cliente", "regiao_vendedor"}
nulos_df["estrategia_imputer"] = nulos_df["feature"].apply(
    lambda f: "SimpleImputer(most_frequent)" if f in _estrategia_cat else "KNNImputer(n_neighbors=5)"
)

print("\n⚠️  Nulos no conjunto de treino:")
if nulos_df.empty:
    print("   Nenhum valor nulo encontrado.")
else:
    exibir_tabela(nulos_df)

# ── 5.4 ColumnTransformer ─────────────────────────────────────────────────────
#
# Intuição do ColumnTransformer: aplica transformações distintas em grupos de
# colunas em PARALELO, estimando parâmetros (mediana, categorias) APENAS no
# treino — prevenindo data leakage ao fit no teste.
#
# Pipeline NUMÉRICO:
#   KNNImputer(k=5)  → imputa pelo valor médio dos 5 vizinhos mais similares.
#                      Superior ao SimpleImputer(median) pois usa a estrutura
#                      multivariada dos dados (correlações entre features).
#   StandardScaler() → z-score (µ=0, σ=1). Necessário para Ridge (regularização
#                      L2 é sensível à escala). Neutro para árvores/boosting.
#
# Pipeline CATEGÓRICO:
#   SimpleImputer(most_frequent) → preenche nulos com a moda da categoria.
#   OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1):
#     → Mapeia cada categoria para um inteiro. Escolha vs. OneHotEncoder:
#       com 71 categorias em product_category, OHE geraria 71+ colunas esparsas.
#       OrdinalEncoder é eficiente para modelos de árvore (que não assumem
#       ordenação — eles splitam por limiar numérico, ignorando a ordem).

numeric_pipe = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler",  StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe,      NUM_FEATURES),
    ("cat", categorical_pipe,  CAT_FEATURES),
], remainder="drop")

print("\n✅ Preprocessor configurado:")
print(f"   Numérico  ({len(NUM_FEATURES)} features): KNNImputer → StandardScaler")
print(f"   Categórico ({len(CAT_FEATURES)} features): SimpleImputer → OrdinalEncoder")


📐 Features: 38 total (32 numéricas + 6 categóricas)
📐 X shape: (95577, 38) | y shape: (95577,)

✂️  Treino: 76,461 | Teste: 19,116

⚠️  Nulos no conjunto de treino:


,feature,pct_nulos_%,estrategia_imputer
0,dias_ate_aprova_h,0.01,KNNImputer(n_neighbors=5)
1,product_weight_g,0.02,KNNImputer(n_neighbors=5)
2,volume_cm3,0.02,KNNImputer(n_neighbors=5)
3,densidade_g_cm3,0.02,KNNImputer(n_neighbors=5)
4,product_photos_qty,1.41,KNNImputer(n_neighbors=5)
5,geolocation_lat,0.28,KNNImputer(n_neighbors=5)
6,geolocation_lng,0.28,KNNImputer(n_neighbors=5)
7,dist_km,0.50,KNNImputer(n_neighbors=5)
8,delta_lat,0.50,KNNImputer(n_neighbors=5)
9,delta_lng,0.50,KNNImputer(n_neighbors=5)



✅ Preprocessor configurado:
   Numérico  (32 features): KNNImputer → StandardScaler
   Categórico (6 features): SimpleImputer → OrdinalEncoder


In [ ]:
# =============================================================================
# SEÇÃO 5 — PRÉ-PROCESSAMENTO
# =============================================================================
 
# ── 5.1 Definição das features por tipo ──────────────────────────────────────
TARGET = "dias_entrega"
 
NUM_FEATURES: List[str] = [
    # ── Temporais ──────────────────────────────────────────────────────────────
    "estimativa_prazo",           # r=+0.43 — feature mais correlacionada
    "dias_ate_aprova_h",          # r=+0.10 — demora na aprovação indica problema
    "dia_semana_compra",          # padrão de dia da semana
    "hora_compra",                # padrão de hora do dia
    "mes_compra",                 # sazonalidade mensal
    "fim_de_semana",              # 1 se sáb/dom — atraso no processamento bancário
    "periodo_dia",                # 0=madrugada, 1=manhã, 2=tarde, 3=noite
    # ── Financeiras ────────────────────────────────────────────────────────────
    "price_total",                # valor total dos produtos
    "freight_total",              # r=+0.18 — frete alto = produto pesado/distante
    "payment_value_total",        # valor total pago (inclui juros)
    "payment_installments_max",   # parcelamento (proxy de valor do pedido)
    "freight_ratio",              # r=+0.10 — frete ÷ preço (proxy de distância)
    "avg_price_per_item",         # ticket médio por item
    "price_range",                # amplitude de preços no pedido
    # ── Itens ──────────────────────────────────────────────────────────────────
    "n_items",                    # quantidade de itens
    "n_sellers",                  # pedidos com múltiplos sellers = mais complexos
    # ── Produto ────────────────────────────────────────────────────────────────
    "product_weight_g",           # r=+0.08 — peso afeta frete e prazo
    "volume_cm3",                 # volume (L×A×C)
    "densidade_g_cm3",            # densidade g/cm³ — tipologia do produto
    "product_photos_qty",         # proxy de qualidade do anúncio
    # ── Geográficas numéricas ──────────────────────────────────────────────────
    "geolocation_lat",            # r=+0.28 — latitude do cliente
    "geolocation_lng",            # longitude do cliente
    "dist_km",                    # r=+0.44 — distância Haversine cliente↔vendedor
    "delta_lat",                  # r=+0.23 — componente Norte-Sul da rota
    "delta_lng",                  # r=+0.12 — componente Leste-Oeste da rota
    "faixa_dist_km",              # r=+0.47 — faixa operacional de distância
    "mesma_uf",                   # r=-0.41 — 1 se cliente e vendedor na mesma UF
    "mesma_regiao",               # r=-0.33 — 1 se mesma macrorregião
    "media_dias_uf_cliente",      # r=+0.46 — target encoding da UF destino
    # ── Seller ─────────────────────────────────────────────────────────────────
    "seller_avg_delivery",        # r=+0.35 — histórico médio do vendedor
    "seller_std_delivery",        # r=+0.15 — variabilidade histórica do vendedor
    "seller_n_orders",            # volume histórico de pedidos do vendedor
]
 
CAT_FEATURES: List[str] = [
    "payment_type",                    # credit_card / boleto / voucher / debit_card
    "customer_state",                  # 27 UFs (proxy de infraestrutura de destino)
    "seller_state",                    # UF do vendedor (proxy de infraestrutura origem)
    "regiao_cliente",                  # Norte / Nordeste / Centro-Oeste / Sudeste / Sul
    "regiao_vendedor",                 # macrorregião do vendedor
    "product_category_name_english",   # 71 categorias de produto
]
 
ALL_FEATURES = NUM_FEATURES + CAT_FEATURES
X = df_clean[ALL_FEATURES].copy()
y = df_clean[TARGET].copy()
 
print(f"\n📐 Features: {len(ALL_FEATURES)} total ({len(NUM_FEATURES)} numéricas + {len(CAT_FEATURES)} categóricas)")
print(f"📐 X shape: {X.shape} | y shape: {y.shape}")
 
# ── 5.2 Split treino/teste ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
print(f"\n✂️  Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")
 
# ── 5.3 Análise de nulos ──────────────────────────────────────────────────────
nulos = X_train.isnull().sum()
nulos_pct = (nulos[nulos > 0] / len(X_train) * 100).round(2)
nulos_df = nulos_pct.reset_index().rename(columns={"index": "feature", 0: "pct_nulos"})
nulos_df["tipo_nulo"] = nulos_df["feature"].map({
    "dias_ate_aprova_h":              "MAR",
    "product_category_name_english":  "MAR",
    "product_weight_g":               "MAR",
    "volume_cm3":                     "MAR",
    "densidade_g_cm3":                "MAR",
    "product_photos_qty":             "MAR",
    "geolocation_lat":                "MAR",
    "geolocation_lng":                "MAR",
    "dist_km":                        "MAR",
    "delta_lat":                      "MAR",
    "delta_lng":                      "MAR",
    "faixa_dist_km":                  "MAR",
}).fillna("MAR")
nulos_df["estrategia"] = nulos_df["feature"].map({
    "product_category_name_english": "SimpleImputer(most_frequent) → OrdinalEncoder",
    "regiao_cliente":                "SimpleImputer(most_frequent) → OrdinalEncoder",
    "regiao_vendedor":               "SimpleImputer(most_frequent) → OrdinalEncoder",
}).fillna("KNNImputer(k=5) → StandardScaler")
 
print("\n⚠️  Nulos no conjunto de treino:")
if nulos_df.empty:
    print("   Sem nulos.")
else:
    exibir_tabela(nulos_df)
 
# ── 5.4 ColumnTransformer ─────────────────────────────────────────────────────
#
# Intuição do ColumnTransformer: aplica transformações distintas em grupos de
# colunas em PARALELO, estimando parâmetros (mediana, categorias) APENAS no
# treino — prevenindo data leakage ao fit no teste.
#
# Pipeline NUMÉRICO:
#   KNNImputer(k=5)  → imputa pelo valor médio dos 5 vizinhos mais similares.
#                      Superior ao SimpleImputer(median) pois usa a estrutura
#                      multivariada dos dados (correlações entre features).
#   StandardScaler() → z-score (µ=0, σ=1). Necessário para Ridge (regularização
#                      L2 é sensível à escala). Neutro para árvores/boosting.
#
# Pipeline CATEGÓRICO:
#   SimpleImputer(most_frequent) → preenche nulos com a moda da categoria.
#   OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1):
#     → Mapeia cada categoria para um inteiro. Escolha vs. OneHotEncoder:
#       com 71 categorias em product_category, OHE geraria 71+ colunas esparsas.
#       OrdinalEncoder é eficiente para modelos de árvore (que não assumem
#       ordenação — eles splitam por limiar numérico, ignorando a ordem).
 
numeric_pipe = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler",  StandardScaler()),
])
 
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
 
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe,      NUM_FEATURES),
    ("cat", categorical_pipe,  CAT_FEATURES),
], remainder="drop")
 


📐 Features: 38 total (32 numéricas + 6 categóricas)
📐 X shape: (95577, 38) | y shape: (95577,)

✂️  Treino: 76,461 | Teste: 19,116

⚠️  Nulos no conjunto de treino:


,feature,pct_nulos,tipo_nulo,estrategia
0,dias_ate_aprova_h,0.01,MAR,KNNImputer(k=5) → StandardScaler
1,product_weight_g,0.02,MAR,KNNImputer(k=5) → StandardScaler
2,volume_cm3,0.02,MAR,KNNImputer(k=5) → StandardScaler
3,densidade_g_cm3,0.02,MAR,KNNImputer(k=5) → StandardScaler
4,product_photos_qty,1.41,MAR,KNNImputer(k=5) → StandardScaler
5,geolocation_lat,0.28,MAR,KNNImputer(k=5) → StandardScaler
6,geolocation_lng,0.28,MAR,KNNImputer(k=5) → StandardScaler
7,dist_km,0.50,MAR,KNNImputer(k=5) → StandardScaler
8,delta_lat,0.50,MAR,KNNImputer(k=5) → StandardScaler
9,delta_lng,0.50,MAR,KNNImputer(k=5) → StandardScaler


In [ ]:
# =============================================================================
# SEÇÃO 6 — BENCHMARK DE MODELOS
# =============================================================================
#
# Modelos selecionados e justificativas:
#
# 1. Ridge (Regressão Linear + L2): baseline interpretável. Verifica se
#    o problema tem componente linear significativo. Coeficientes diretamente
#    interpretáveis. Regularização L2 lida com multicolinearidade entre
#    features de preço.
#
# 2. Decision Tree: baseline não-linear simples. Interpretável por regras.
#    Confirma se existem segmentações naturais nos dados (ex: "pedidos para
#    AM com peso > 5kg → prazo longo").
#
# 3. Random Forest: ensemble bagging — treina N árvores em subsets aleatórios
#    de dados E features, depois agrega por média. Reduz variância sem aumentar
#    viés. Robusto a outliers nas features.
#
# 4. Gradient Boosting: boosting sequencial — cada árvore corrige os resíduos
#    da anterior, minimizando a loss via gradiente descendente. Captura padrões
#    não-lineares complexos e interações entre features.
#
# 5. XGBoost: boosting otimizado com regularização L1/L2, shrinkage e column
#    subsampling. Estado da arte em dados tabulares estruturados. Lida com
#    missing values nativamente (aprende o melhor caminho para NaN).
 
def avaliar_regressao(nome: str, y_true, y_pred: np.ndarray) -> Dict:
    """
    Calcula RMSE, MAE, MAPE e R² para um modelo de regressão.
    
    Intuição das métricas:
    - RMSE: √MSE — penaliza erros grandes quadraticamente. Mesma unidade do target.
    - MAE:  erro médio absoluto — robusto a outliers. "Em média, erra X dias."
    - MAPE: erro percentual — interpretabilidade para o negócio.
    - R²:   proporção da variância explicada (1 = perfeito, 0 = baseline naive).
    """
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    mape = float(np.mean(np.abs((np.array(y_true) - y_pred) / np.clip(y_true, 1, None))) * 100)
    r2   = float(r2_score(y_true, y_pred))
    return {"modelo": nome, "RMSE": rmse, "MAE": mae, "MAPE_%": mape, "R2": r2}
 
 
MODELOS_PIPELINE: Dict[str, Pipeline] = {
    "Ridge": Pipeline([
        ("pre", preprocessor),
        ("reg", Ridge(alpha=10.0))
    ]),
    "DecisionTree": Pipeline([
        ("pre", preprocessor),
        ("reg", DecisionTreeRegressor(max_depth=8, min_samples_leaf=50, random_state=RANDOM_STATE))
    ]),
    "RandomForest": Pipeline([
        ("pre", preprocessor),
        ("reg", RandomForestRegressor(n_estimators=150, max_depth=12,
                                      min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    "GradientBoosting": Pipeline([
        ("pre", preprocessor),
        ("reg", GradientBoostingRegressor(n_estimators=150, learning_rate=0.08,
                                          max_depth=5, subsample=0.8,
                                          random_state=RANDOM_STATE))
    ]),
    "XGBoost": Pipeline([
        ("pre", preprocessor),
        ("reg", XGBRegressor(n_estimators=200, learning_rate=0.08, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
    ]),
}
 
resultados = []
print("\n" + "="*65)
print("BENCHMARK — TREINAMENTO E AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*65)
 
for nome, pipe in MODELOS_PIPELINE.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    met = avaliar_regressao(nome, y_test, y_pred)
    resultados.append(met)
    print(f"  {nome:20s} | RMSE={met['RMSE']:.3f} | MAE={met['MAE']:.3f} | R²={met['R2']:.4f}")
 
resultado_df = pd.DataFrame(resultados).set_index("modelo").sort_values("RMSE")
print("\n📊 Ranking final")
exibir_tabela(resultado_df.round(4))
resultado_df.round(4).to_csv(OUT / "benchmark_resultados.csv")
 
 
# ── Gráfico de comparação ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
modelos_l = resultado_df.index.tolist()
rmses = resultado_df["RMSE"].values
r2s   = resultado_df["R2"].values
cores = ["#16a34a" if i == 0 else PALETA for i in range(len(modelos_l))]
 
axes[0].barh(modelos_l[::-1], rmses[::-1], color=cores[::-1], edgecolor="white", alpha=0.9)
for i, v in enumerate(rmses[::-1]):
    axes[0].text(v + 0.02, i, f"{v:.3f}", va="center", fontsize=9)
axes[0].set_title("RMSE — Teste (menor = melhor)"); axes[0].set_xlabel("RMSE (dias)")
 
axes[1].barh(modelos_l[::-1], r2s[::-1], color=cores[::-1], edgecolor="white", alpha=0.9)
for i, v in enumerate(r2s[::-1]):
    axes[1].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=9)
axes[1].set_title("R² — Teste (maior = melhor)"); axes[1].set_xlabel("R²"); axes[1].set_xlim(0, 1.05)
 
plt.suptitle("Comparação de Modelos de Regressão — Olist", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "fig6_benchmark.png", bbox_inches="tight")
plt.show()
 


BENCHMARK — TREINAMENTO E AVALIAÇÃO NO CONJUNTO DE TESTE
  Ridge                | RMSE=6.161 | MAE=4.384 | R²=0.3758
  DecisionTree         | RMSE=6.009 | MAE=4.269 | R²=0.4062
  RandomForest         | RMSE=5.732 | MAE=4.036 | R²=0.4598
  GradientBoosting     | RMSE=5.669 | MAE=3.984 | R²=0.4715
  XGBoost              | RMSE=5.620 | MAE=3.936 | R²=0.4806

📊 Ranking final


,RMSE,MAE,MAPE_%,R2
modelo,,,,
XGBoost,5.6202,3.9361,45.6789,0.4806
GradientBoosting,5.6694,3.9839,46.6358,0.4715
RandomForest,5.7317,4.0365,47.9259,0.4598
DecisionTree,6.0093,4.2694,51.1057,0.4062
Ridge,6.1613,4.3843,52.6398,0.3758


In [ ]:
# =============================================================================
# SEÇÃO 7 — AJUSTE DE HIPERPARÂMETROS (RandomizedSearchCV — XGBoost)
# =============================================================================
#
# Intuição do RandomizedSearchCV:
# Amostra N combinações aleatórias do espaço de hiperparâmetros (Bergstra & Bengio,
# 2012). Para espaços grandes, é ~100x mais eficiente que GridSearchCV com
# desempenho similar. Com n_iter=20 e cv=5: 100 treinamentos totais.
#
# Hiperparâmetros e seus efeitos:
# • n_estimators:      mais árvores → menos variância, mais custo computacional
# • learning_rate:     shrinkage — taxa de contribuição de cada árvore (menor = mais robusto)
# • max_depth:         controla complexidade da árvore individual (maior = mais overfit)
# • subsample:         fração de linhas por árvore (bagging — reduz overfitting)
# • colsample_bytree:  fração de features por árvore (regulariza, análogo ao Random Forest)
# • reg_alpha (L1):    regularização Lasso — promove esparsidade de features
# • reg_lambda (L2):   regularização Ridge — suaviza pesos
 
print("\n" + "="*60)
print("AJUSTE DE HIPERPARÂMETROS — XGBoost (RandomizedSearchCV)")
print("="*60)
 
xgb_pipe_tunavel = Pipeline([
    ("pre", preprocessor),
    ("reg", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
])
 
param_dist = {
    "reg__n_estimators":     [150, 200, 300, 400],
    "reg__learning_rate":    [0.03, 0.05, 0.08, 0.1, 0.15],
    "reg__max_depth":        [4, 5, 6, 7],
    "reg__subsample":        [0.7, 0.8, 0.9, 1.0],
    "reg__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "reg__min_child_weight": [1, 3, 5, 10],
    "reg__reg_alpha":        [0, 0.01, 0.1, 1.0],
    "reg__reg_lambda":       [0.5, 1.0, 2.0, 5.0],
}
 
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
 
random_search = RandomizedSearchCV(
    estimator=xgb_pipe_tunavel,
    param_distributions=param_dist,
    n_iter=20,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
 
random_search.fit(X_train, y_train)
 
best_params_df = pd.DataFrame({
    "param": [p.replace("reg__", "") for p in random_search.best_params_.keys()],
    "value": list(random_search.best_params_.values())
})
print(f"\n✅ Melhores parâmetros encontrados:")
exibir_tabela(best_params_df)
print(f"\n   RMSE CV (melhor): {-random_search.best_score_:.4f} dias")
 
# Avaliação final no teste
y_pred_tunado = random_search.predict(X_test)
met_tunado = avaliar_regressao("XGBoost (tunado)", y_test, y_pred_tunado)

tunado_df = pd.DataFrame([{
    "modelo": "XGBoost (tunado)",
    "RMSE": met_tunado["RMSE"],
    "MAE": met_tunado["MAE"],
    "R2": met_tunado["R2"],
}]).set_index("modelo")
print(f"\n📊 XGBoost tunado no teste:")
exibir_tabela(tunado_df.round(4))


AJUSTE DE HIPERPARÂMETROS — XGBoost (RandomizedSearchCV)
Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:
# =============================================================================
# SEÇÃO 8 — ANÁLISE DE RESÍDUOS E FEATURE IMPORTANCE
# =============================================================================
 
# ── 8.1 Resíduos ───────────────────────────────────────────────────────────────
y_pred_best = MODELOS_PIPELINE["XGBoost"].predict(X_test)
residuos = y_test.values - y_pred_best
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
axes[0].scatter(y_pred_best, y_test.values, alpha=0.1, s=4, color=PALETA)
axes[0].plot([0, 46], [0, 46], "r--", lw=1.5, label="Predição Perfeita")
axes[0].set_xlabel("Predito (dias)"); axes[0].set_ylabel("Real (dias)")
axes[0].set_title("Real vs. Predito — XGBoost"); axes[0].legend()
 
axes[1].scatter(y_pred_best, residuos, alpha=0.1, s=4, color=PALETA)
axes[1].axhline(0, color="red", ls="--", lw=1.5)
axes[1].set_xlabel("Predito (dias)"); axes[1].set_ylabel("Resíduo (dias)")
axes[1].set_title("Resíduos vs. Predito")
 
axes[2].hist(residuos, bins=50, color=PALETA, edgecolor="white", alpha=0.9)
axes[2].axvline(0, color="red", ls="--", lw=1.5)
axes[2].set_xlabel("Resíduo (dias)"); axes[2].set_ylabel("Frequência")
axes[2].set_title(f"Distribuição de Resíduos | Bias={residuos.mean():.3f}d")
 
plt.tight_layout()
plt.savefig(OUT / "fig7_residuos.png", bbox_inches="tight")
plt.show()
 
# ── 8.2 Feature Importance ─────────────────────────────────────────────────────
fi = MODELOS_PIPELINE["XGBoost"].named_steps["reg"].feature_importances_
fi_df = (pd.DataFrame({"feature": ALL_FEATURES, "importance": fi})
           .sort_values("importance", ascending=True)
           .tail(15))
 
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df["feature"], fi_df["importance"], color=PALETA, edgecolor="white", alpha=0.85)
ax.set_title("Feature Importance — XGBoost (Top 15 por Gain médio)")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(OUT / "fig8_feature_importance.png", bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# SEÇÃO 9 — RESUMO EXECUTIVO
# =============================================================================
 
best_modelo = resultado_df.iloc[0]

dataset_summary = pd.DataFrame([
    {"Métrica": "Pedidos entregues (após limpeza)", "Valor": f"{len(df_clean):,}"},
    {"Métrica": "Features numéricas", "Valor": len(NUM_FEATURES)},
    {"Métrica": "Features categóricas", "Valor": len(CAT_FEATURES)},
    {"Métrica": "Total de features", "Valor": len(ALL_FEATURES)},
    {"Métrica": "Target médio (dias)", "Valor": f"{df_clean['dias_entrega'].mean():.1f}"},
    {"Métrica": "Target mediana (dias)", "Valor": f"{df_clean['dias_entrega'].median():.0f}"},
    {"Métrica": "Target std (dias)", "Valor": f"{df_clean['dias_entrega'].std():.1f}"},
])
print("\n📌 Resumo do dataset")
exibir_tabela(dataset_summary)

final_model_summary = pd.DataFrame([
    {"Métrica": "Modelo final", "Valor": "XGBoost (tunado via RandomizedSearchCV)"},
    {"Métrica": "RMSE (teste)", "Valor": f"{met_tunado['RMSE']:.3f}"},
    {"Métrica": "MAE (teste)", "Valor": f"{met_tunado['MAE']:.3f}"},
    {"Métrica": "R² (teste)", "Valor": f"{met_tunado['R2']:.4f}"},
    {"Métrica": "Variância explicada", "Valor": f"{met_tunado['R2']*100:.1f}%"},
])
print("\n📌 Resumo executivo")
exibir_tabela(final_model_summary)

feature_impact = corr_target.head(5).reset_index().rename(columns={"index": "feature", "dias_entrega": "pearson_r"})
feature_impact["pearson_r"] = feature_impact["pearson_r"].round(3)
print("\n📌 Features mais impactantes")
exibir_tabela(feature_impact)

future_improvements = pd.DataFrame({
    "Melhoria": [
        "Distância Haversine (lat/lng cliente vs. vendedor)",
        "Histórico de desempenho do vendedor (% atrasos)",
        "Separação de modelos por macrorregião geográfica",
        "SHAP values para explicabilidade por previsão individual",
        "Stacking: XGBoost + LightGBM + RandomForest",
    ]
})
print("\n📌 Melhorias futuras")
exibir_tabela(future_improvements)

logger.info(f"✅ Pipeline concluído. Outputs salvos em {OUT}")
 


📌 Resumo do dataset


,Métrica,Valor
0,Pedidos entregues (após limpeza),"95,577"
1,Features numéricas,16
2,Features categóricas,4
3,Total de features,20
4,Target médio (dias),11.6
5,Target mediana (dias),10
6,Target std (dias),7.8



📌 Resumo executivo


,Métrica,Valor
0,Modelo final,XGBoost (tunado via RandomizedSearchCV)
1,RMSE (teste),5.820
2,MAE (teste),4.105
3,R² (teste),0.4430
4,Variância explicada,44.3%



📌 Features mais impactantes


,feature,pearson_r
0,faixa_dist_km,0.471
1,media_dias_uf_cliente,0.461
2,dist_km,0.441
3,estimativa_prazo,0.430
4,mesma_uf,-0.413



📌 Melhorias futuras


,Melhoria
0,Distância Haversine (lat/lng cliente vs. vende...
1,Histórico de desempenho do vendedor (% atrasos)
2,Separação de modelos por macrorregião geográfica
3,SHAP values para explicabilidade por previsão ...
4,Stacking: XGBoost + LightGBM + RandomForest


INFO | ✅ Pipeline concluído. Outputs salvos em ..\dataframes\processed
